# 07 — Gerar e conferir o artigo em LaTeX

Este notebook lê os resultados atualizados dos notebooks analíticos, gera `artigo.tex` e `referencias.bib`, compila o PDF e verifica extensão e referências.

O manual pede até três páginas, pelo menos uma figura e respostas sobre problema, métodos, resultados e conclusões, incluindo limitações.

**Escolhas registradas:** distância mínima total pelas ligações e ruas; L3 descritiva sem ajuste de cauda; modelo simples de renda sem efeitos fixos; sem CCA, percolação ou análise de favela. O autor e a afiliação são Lia Sucupira — Universidade de Fortaleza.

**Atualização desta versão:** a regra de distância mudou para minimizar o percurso completo. O grupo do PA Ilha de Maré fica fora apenas dos ajustes L2 e renda porque seus setores não passam pelo filtro de ligação à rua; ele continua no cadastro municipal. O notebook de diagnóstico e os valores atualizados registram essa exceção.

In [1]:
from pathlib import Path
import json
import re
import shutil
import subprocess

import pandas as pd
from IPython.display import display, Markdown, IFrame

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "PLANO.md").exists())
ARTIGO = RAIZ / "artigo"
ARTIGO.mkdir(exist_ok=True)
BASE = RAIZ / "dados" / "processados" / "analises"

# Preencher com as respostas da autora, sem inferir nomes.
AUTOR = "Lia Sucupira"
AFILIACAO = "Universidade de Fortaleza"
L3_DESCRITIVA_CONFIRMADA = True

l1 = pd.read_csv(BASE/"L1/resultados_L1.csv").set_index("versao")
l2 = pd.read_csv(BASE/"L2/resultados_L2.csv").set_index("versao")
renda = pd.read_csv(BASE/"renda/resultados_renda.csv").set_index("versao")
distancias = pd.read_csv(BASE/"distancias/resumo_capitais.csv")
calor = pd.read_csv(BASE/"distancias/densidade_renda_sus.csv").set_index(["q_densidade","q_renda"])
grupo16 = pd.read_csv(RAIZ/"dados/processados/retomada/mapas_grupos_P90/grupos_16_renda_densidade.csv")
p90 = pd.read_csv(RAIZ/"dados/processados/retomada/mapas_grupos_P90/limites_P90_por_capital.csv")
pobre = grupo16.loc[(grupo16.versao=="SUS")&(grupo16.q_densidade==1)&(grupo16.q_renda==1)].iloc[0]
p90_fortaleza = p90.loc[(p90.capital=="Fortaleza - CE")&(p90.versao=="SUS")].iloc[0]
assert (RAIZ/"figuras/final/figura_principal.pdf").exists()
assert shutil.which("pdflatex") and shutil.which("bibtex") and shutil.which("latexmk")
pendencias = []
if not AUTOR or not AFILIACAO:
    pendencias.append("autoria e afiliação")
if not L3_DESCRITIVA_CONFIRMADA:
    pendencias.append("confirmação da apresentação descritiva da L3")
print("Situação:", "rascunho; pendências: "+", ".join(pendencias) if pendencias else "configurações preenchidas")
display(renda[["delta","delta_ic95_inferior","delta_ic95_superior"]].round(3))


Situação: configurações preenchidas


,delta,delta_ic95_inferior,delta_ic95_superior
versao,,,
SUS,-0.196,-0.353,-0.029
Todos,0.029,-0.122,0.188


## 1. Referências verificadas nos arquivos do projeto

As referências teóricas foram conferidas nos PDFs locais. As fontes de dados CNES/OpenDataSUS, IBGE e OpenStreetMap/Geofabrik são identificadas nos métodos.

O artigo usa a L3 de forma descritiva, sem estimar expoente de cauda. O modelo de renda é o modelo conjunto simples escolhido pela autora. Não comparamos localizações reais com uma solução otimizada.

In [2]:
bibliografia = r"""@book{bettencourt2021,
  author = {Bettencourt, Lu{\'i}s M. A.},
  title = {Introduction to Urban Science: Evidence and Theory of Cities as Complex Systems},
  publisher = {The MIT Press},
  year = {2021}
}
@article{gastner2006,
  author = {Gastner, Michael T. and Newman, M. E. J.},
  title = {Optimal design of spatial distribution networks},
  journal = {Physical Review E},
  volume = {74},
  pages = {016117},
  year = {2006},
  doi = {10.1103/PhysRevE.74.016117}
}
@article{xu2020,
  author = {Xu, Yanyan and Olmos, Luis E. and Abbar, Sofiane and Gonz{\'a}lez, Marta C.},
  title = {Deconstructing laws of accessibility and facility distribution in cities},
  journal = {Science Advances},
  volume = {6},
  pages = {eabb4112},
  year = {2020},
  doi = {10.1126/sciadv.abb4112}
}
@article{melo2014,
  author = {Melo, Hygor Piaget M. and Moreira, Andr{\'e} A. and Batista, {\'E}lcio and Makse, Hern{\'a}n A. and Andrade, Jos{\'e} S.},
  title = {Statistical Signs of Social Influence on Suicides},
  journal = {Scientific Reports},
  volume = {4},
  pages = {6239},
  year = {2014},
  doi = {10.1038/srep06239}
}
@article{newman2005,
  author = {Newman, M. E. J.},
  title = {Power laws, Pareto distributions and {Zipf}'s law},
  journal = {Contemporary Physics},
  volume = {46},
  pages = {323--351},
  year = {2005}
}
@article{maia2023,
  author = {Maia, Gabriel and Ponte, Caio and Caminha, Carlos and Furtado, Lara S. and Melo, Hygor P. M. and Furtado, Vasco},
  title = {A global empirical study on how street networks facilitate driving longer distances},
  journal = {Scientific Reports},
  volume = {13},
  pages = {18154},
  year = {2023},
  doi = {10.1038/s41598-023-45236-7}
}
@article{arcaute2015,
  author = {Arcaute, Elsa and Hatna, Erez and Ferguson, Peter and Youn, Hyejin and Johansson, Anders and Batty, Michael},
  title = {Constructing cities, deconstructing scaling laws},
  journal = {Journal of the Royal Society Interface},
  volume = {12},
  pages = {20140745},
  year = {2015},
  doi = {10.1098/rsif.2014.0745}
}
"""
(ARTIGO/"referencias.bib").write_text(bibliografia,encoding="utf-8")
print("Bibliografia local escrita:",ARTIGO/"referencias.bib")


Bibliografia local escrita: /home/liasucf/ciencia-de-cidades/artigo/referencias.bib


## 2. Texto do artigo e figura de síntese

A figura atual tem quatro painéis: os mapas de calor dos 16 grupos, áreas de atendimento em Fortaleza e setores acima do P90 dessa capital. O texto abaixo é preenchido diretamente dos CSVs atualizados; não digitamos resultados à mão.

A redação evita tratar associação como causa, o expoente como prova de localização ótima ou o P90 como limite clínico.

In [3]:
def numero(valor, casas=3):
    return f"{valor:.{casas}f}".replace(".",r"{,}")

def intervalo(linha, inferior, superior, casas=3):
    return f"[{numero(linha[inferior],casas)};\\,{numero(linha[superior],casas)}]"

def escapar_latex(texto):
    especiais = {"\\":r"\textbackslash{}", "&":r"\&", "%":r"\%", "$":r"\$",
                 "#":r"\#", "_":r"\_", "{":r"\{", "}":r"\}", "~":r"\textasciitilde{}",
                 "^":r"\textasciicircum{}"}
    return "".join(especiais.get(c,c) for c in texto)

s = distancias.loc[distancias.versao == "SUS"]
n_setores_validos = int(s.n_setores.sum())
populacao_valida = int(s.populacao.sum())
autor_latex = escapar_latex(AUTOR) if AUTOR else "Autoria a informar"
afiliacao_latex = escapar_latex(AFILIACAO) if AFILIACAO else "Afiliação a informar"
estado_latex = "Rascunho para revisão — campos e escolhas pendentes" if pendencias else ""

template = r"""\documentclass[10pt,a4paper,twocolumn]{article}
\usepackage[brazil]{babel}
\usepackage[utf8]{inputenc}
\usepackage[T1]{fontenc}
\usepackage[margin=2cm,columnsep=0.65cm]{geometry}
\usepackage{graphicx,amsmath,microtype,url}
\usepackage[numbers,sort&compress]{natbib}
\usepackage[font=footnotesize,labelfont=bf]{caption}
\usepackage{titlesec}
\titleformat{\section}{\normalfont\bfseries\normalsize}{\thesection.}{.4em}{}
\titlespacing*{\section}{0pt}{1.1ex plus .2ex}{.5ex}
\setlength{\parindent}{1em}
\setlength{\parskip}{0pt}
\setlength{\textfloatsep}{8pt}
\setlength{\dbltextfloatsep}{8pt}
\setlength{\floatsep}{6pt}
\setlength{\bibsep}{1pt}
\title{\vspace{-1.4em}\large\bfseries Escala e desigualdades no acesso estimado a pronto-atendimentos nas capitais brasileiras}
\author{\normalsize @AUTOR@\\[-1pt]\small @AFILIACAO@}
\date{}
\begin{document}
\maketitle
\vspace{-1.2em}

\section{Problema}
Contagens municipais podem esconder diferenças de acesso entre bairros. Investigamos como a oferta de pronto-atendimentos varia com população e renda e como se distribuem as distâncias estimadas pelas ruas nas capitais brasileiras. No cadastro CNES, consideramos estabelecimentos ativos do tipo 73 que atendem o SUS, incluindo UPAs, PAs, AMA 24h e SPAs.

Leis de escala relacionam indicadores urbanos à população \citep{bettencourt2021}. Para serviços públicos, um modelo de otimização prevê densidade de unidades proporcional à densidade populacional elevada a $2/3$ \citep{gastner2006}. Essa previsão depende de hipóteses e é usada aqui como referência, não como prova de que os serviços reais estejam bem ou mal localizados. Estudos intraurbanos também destacam a relação entre distribuição de serviços, população e rede viária \citep{xu2020}.

\section{Dados e métodos}
Integramos o cadastro CNES/OpenDataSUS (24/10/2026), população, setores e renda média do responsável pelo domicílio do Censo 2022 (IBGE), e ruas do OpenStreetMap/Geofabrik. A renda não é per capita. Há 245 unidades SUS e 274 unidades do tipo 73 nas capitais; Aracaju, sem unidade desse tipo no recorte, não entra nos ajustes. As regressões usam as outras 26 capitais. Corrigimos coordenadas e reunimos polígonos repetidos: cada setor tem população e área oficial contadas uma vez.

Representamos cruzamentos por nós e trechos dirigidos por arestas. Para cada setor, o Dijkstra encontra o menor custo total até uma unidade: ligação do ponto do setor à rua + percurso pela rede + ligação da unidade à rua. Assim, a unidade escolhida minimiza a soma inteira, não apenas o trecho entre nós. Distância é estimada para carro e não equivale a tempo de viagem. Excluímos setores sem moradores, sem rota finita ou com ligação do ponto à rua acima de 2 km; restam @NSETORES@ setores e @POPULACAO@ moradores. Em Todos, o PA Ilha de Maré está no cadastro, mas seus dois setores (1.178 moradores) não passam pelo filtro de 2 km; sem área válida, não entram nos ajustes de densidade.

Ajustamos por mínimos quadrados em logaritmos: (L1) número de unidades por capital em função da população; (L2) densidade de unidades $D_j=m_j/A_j$ em função da densidade de moradores $\rho_j=P_j/A_j$ na área de atendimento $j$; e um modelo conjunto que inclui renda média $\bar r_j$, com coeficiente $\delta$. Áreas são definidas pelos setores atribuídos à unidade mais acessível; unidades ligadas ao mesmo nó compartilham uma área e são contadas separadamente em $m_j$. Há 245 áreas e unidades em SUS; em Todos, 272 áreas representam 273 unidades (uma das 274 não tem setor válido associado). O modelo de renda reúne capitais, sem efeitos fixos. Intervalos percentis de 95\% usam 2.000 reamostragens bootstrap de capitais (L1) ou áreas (L2 e renda), semente 42 \citep{melo2014}.

Por fim, calculamos curvas complementares de distância e Gini ponderados pela população. Para comparar capitais, dividimos distância pela média da própria cidade. Quartis de renda e densidade são definidos dentro de cada capital. A L3 é descritiva, sem estimar expoente de cauda \citep{newman2005}.

\begin{figure*}[t]
\centering
\includegraphics[width=0.97\textwidth]{../figuras/final/figura_principal.pdf}
\caption{Acesso estimado ao pronto-atendimento SUS. (a) Distância relativa média nos 16 grupos de renda e densidade, agregados entre capitais; 1 equivale à média da cidade. (b) Moradores em setores com distância igual ou acima do P90 de cada capital; o corte é relativo à própria cidade. (c) Áreas estimadas de atendimento em Fortaleza (12 grupos). (d) Setores de Fortaleza com distância igual ou acima do P90 SUS, 5,73 km. Quartis são calculados por capital.}
\label{fig:principal}
\end{figure*}

\section{Resultados e conclusão}
Na L1, $\beta=@BETA_SUS@$ (IC 95\% $@IC_BETA_SUS@$); o intervalo inclui 1, portanto o ajuste é compatível com proporcionalidade entre população e unidades. Com todos os estabelecimentos, $\beta=@BETA_TODOS@$ ($@IC_BETA_TODOS@$).

Na L2, $\alpha=@ALPHA_SUS@$ ($@IC_ALPHA_SUS@$) em SUS e $@ALPHA_TODOS@$ ($@IC_ALPHA_TODOS@$) em Todos. Os intervalos ficam acima de $2/3$ e abaixo de 1: a relação observada aumenta mais com a densidade de moradores que a referência de dois terços, mas menos que proporcionalmente. Isso não demonstra, por si só, ineficiência nem localização ótima.

No modelo de renda SUS, $\delta=@DELTA_SUS@$ ($@IC_DELTA_SUS@$). Mantida a densidade populacional, renda duas vezes maior se associa a oferta prevista @FATOR_DELTA@ vez a oferta de referência, cerca de 13\% menor. Em Todos, $\delta=@DELTA_TODOS@$ ($@IC_DELTA_TODOS@$), com intervalo que inclui zero; não há associação clara nesse recorte. São associações, não efeitos causais, e o modelo mistura diferenças dentro e entre capitais.

A tabela de renda e densidade revela uma situação que a média conjunta não resume: no grupo de menor renda e menor densidade, a distância relativa média é @POBRE_RALO@ e @POBRE_P90@\% dos moradores estão em setores acima do P90 de suas capitais. Em Fortaleza, o P90 SUS é 5,73 km e @P90_FORTALEZA_PCT@\% da população válida está em setores nesse corte. Como os moradores são contados por setor, essa fração pode passar de 10\%. P90 é um marcador relativo para identificar as maiores distâncias de cada cidade, não um padrão clínico. As curvas e o Gini descrevem a distribuição das distâncias, sem teste formal de lei de potência.

Os resultados mostram por que contagens municipais e um único expoente não bastam para descrever o acesso: as distâncias variam entre grupos sociais e cidades. O expoente L2 reúne áreas de 26 capitais e não descreve necessariamente a relação dentro de cada cidade; além disso, $D_j$ e $\rho_j$ compartilham a área no denominador, o que pode gerar associação matemática. Os intervalos reamostram áreas, sem modelar explicitamente o agrupamento por capital nem a dependência espacial, e devem ser lidos como incerteza exploratória. As estimativas também dependem dos pontos representativos dos setores, das ligações à rua, da cobertura da rede OSM, dos limites municipais e do corte de 2 km. Estabelecimentos são contados sem informação de capacidade, demanda, qualidade ou tempo real de viagem; 1.178 moradores dos setores associados ao PA Ilha de Maré ficam fora das análises de distância por esse filtro. A renda é agregada e não mede diretamente necessidade clínica. Portanto, identificamos padrões e diferenças de acesso estimado, mas não provamos que uma unidade esteja em localização ruim nem que a renda cause as diferenças.

{\footnotesize
\bibliographystyle{unsrtnat}
\bibliography{referencias}
}
\end{document}
"""
substituicoes = {
    "@AUTOR@":autor_latex, "@AFILIACAO@":afiliacao_latex, "@ESTADO@":estado_latex,
    "@BETA_SUS@":numero(l1.loc["SUS","beta"]), "@BETA_TODOS@":numero(l1.loc["Todos","beta"]),
    "@IC_BETA_SUS@":intervalo(l1.loc["SUS"],"ic95_inferior","ic95_superior"),
    "@IC_BETA_TODOS@":intervalo(l1.loc["Todos"],"ic95_inferior","ic95_superior"),
    "@ALPHA_SUS@":numero(l2.loc["SUS","alpha"]), "@ALPHA_TODOS@":numero(l2.loc["Todos","alpha"]),
    "@IC_ALPHA_SUS@":intervalo(l2.loc["SUS"],"ic95_inferior","ic95_superior"),
    "@IC_ALPHA_TODOS@":intervalo(l2.loc["Todos"],"ic95_inferior","ic95_superior"),
    "@FATOR_ALPHA@":numero(l2.loc["SUS","fator_D_ao_dobrar_rho"],2),
    "@DELTA_SUS@":numero(renda.loc["SUS","delta"]), "@DELTA_TODOS@":numero(renda.loc["Todos","delta"]),
    "@IC_DELTA_SUS@":intervalo(renda.loc["SUS"],"delta_ic95_inferior","delta_ic95_superior"),
    "@IC_DELTA_TODOS@":intervalo(renda.loc["Todos"],"delta_ic95_inferior","delta_ic95_superior"),
    "@FATOR_DELTA@":numero(renda.loc["SUS","fator_D_renda_dobra"],2),
    "@NSETORES@":f'{n_setores_validos:,}'.replace(",",r"\,"),
    "@POPULACAO@":f'{populacao_valida:,}'.replace(",",r"\,"),
    "@POBRE_RALO@":numero(calor.loc[(1,1),"d_rel_media"],2),
    "@POBRE_P90@":numero(pobre.parcela_muito_longe_pct,1),
    "@P90_FORTALEZA_PCT@":numero(p90_fortaleza.parcela_muito_longe_pct,1),
}
texto = template
for marcador,valor in substituicoes.items():
    texto = texto.replace(marcador,valor)
assert not re.search(r"@[A-Z_]+@",texto), "Marcador não preenchido."
(ARTIGO/"artigo.tex").write_text(texto,encoding="utf-8")
(ARTIGO/"estado_rascunho.json").write_text(
    json.dumps({"autor":AUTOR,"afiliacao":AFILIACAO,"L3_descritiva_confirmada":L3_DESCRITIVA_CONFIRMADA,
                "pendencias":pendencias},ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
print("Texto gerado:",ARTIGO/"artigo.tex")


Texto gerado: /home/liasucf/ciencia-de-cidades/artigo/artigo.tex


## 3. Compilar e conferir o limite de páginas

Executamos LaTeX e BibTeX por `latexmk`. Depois verificamos o número de páginas, citações não resolvidas e caixas que ultrapassam a largura. A inspeção visual do PDF também é necessária para conferir legibilidade.

In [4]:
processo = subprocess.run(
    ["latexmk","-pdf","-interaction=nonstopmode","-halt-on-error","artigo.tex"],
    cwd=ARTIGO,capture_output=True,text=True,encoding="utf-8",errors="replace",timeout=180
)
(ARTIGO/"compilacao.log").write_text(processo.stdout+"\n"+processo.stderr,encoding="utf-8")
assert processo.returncode == 0, processo.stdout[-5000:]+processo.stderr[-2000:]
info = subprocess.check_output(["pdfinfo",str(ARTIGO/"artigo.pdf")],text=True)
paginas = int(re.search(r"Pages:\s+(\d+)",info).group(1))
log = (ARTIGO/"artigo.log").read_text(errors="replace")
texto_pdf = subprocess.check_output(["pdftotext",str(ARTIGO/"artigo.pdf"),"-"],text=True)
assert paginas <= 3, f"PDF com {paginas} páginas: revisar o texto."
assert "undefined citations" not in log and "There were undefined references" not in log
assert "??" not in texto_pdf, "Conferir referência não resolvida."
print("Páginas:",paginas,"(limite: 3)")
print("Referências resolvidas: sim")
print("Avisos de caixa larga:",log.count("Overfull \\hbox"))
print("Estado:",", ".join(pendencias) if pendencias else "campos preenchidos")
display(Markdown("[Abrir o PDF do artigo](../artigo/artigo.pdf)"))
(ARTIGO/"verificacao.json").write_text(json.dumps({
    "paginas":paginas,"limite_paginas":3,"referencias_resolvidas":True,
    "avisos_overfull_hbox":log.count("Overfull \\hbox"),
    "rascunho_com_pendencias":bool(pendencias),"pendencias":pendencias
},ensure_ascii=False,indent=2)+"\n",encoding="utf-8")


Páginas: 2 (limite: 3)
Referências resolvidas: sim
Avisos de caixa larga: 0
Estado: campos preenchidos


[Abrir o PDF do artigo](../artigo/artigo.pdf)

161

## 4. Conferência final com a autora

- Autoria e afiliação: Lia Sucupira — Universidade de Fortaleza.
- Distância pela soma completa: aprovada e aplicada.
- L3 descritiva: aprovada, sem ajuste de cauda.
- Conferir visualmente a figura no PDF e conversar sobre a interpretação dos resultados e limitações antes da entrega.